In [ ]:
import yfinance as yf
import pandas as pd
import requests
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
data = "GC=F, SI=F, DX-Y.NYB"
atr_list = data.split(', ')

#for S&P ^GSPC

In [ ]:
print(atr_list) 

In [ ]:
Capital = 20000
Risk = 0.02
Risk_Amount = Capital * Risk


#### Download data from Yahoo Finance

In [ ]:
data_recent = yf.download(data, start="2023-01-01", end="2023-12-31", interval="1d")

In [ ]:
data_recent.dropna(inplace = True)

In [ ]:
data_recent.head(3)

#### Calculate ATR for Position Sizing

In [ ]:
#Reorentate the stacked headers into columns
data_recent_atr = data_recent.stack(level=1).reset_index(level=0).rename({'level_1': 'Ticker'}, axis=1)
data_recent_atr.reset_index(inplace=True)
data_recent_atr.rename(columns = {'index':'Ticker'}, inplace=True)
data_recent_atr.sort_values(['Ticker', 'Date'], inplace = True)

In [ ]:
#Set up a data frame to append the ATR values to later on
atr_frame = pd.DataFrame(columns = atr_list)

In [ ]:
#function to calculate ART based on 20 day average

'''
def calculate_atr(data, atr_period=20):
    
    for Ticker in data:
        data['High-Low'] = data['High'] - data['Low']
        data['High-PrevClose'] = abs(data['High'] - data['Close'].shift(1))
        data['Low-PrevClose'] = abs(data['Low'] - data['Close'].shift(1))
        data['TR'] = data[['High-Low', 'High-PrevClose', 'Low-PrevClose']].max(axis=1) #finds the max value of the three value
        data['ATR'] = data['TR'].rolling(window=atr_period).mean()

    return data

'''

In [ ]:
#calculate ATR and assign to new df
atr_df = calculate_atr(data_recent_atr)

In [ ]:
#Add ATRs to new frame for use in calculations later.
'''

ATRs = atr_df.groupby("Ticker")["ATR"].last()
atr_frame = pd.concat([atr_frame, ATRs], ignore_index=True)
atr_frame.tail()

'''



#### Calculate Returns

In [ ]:
#requires data downloaded from yf
#requites a list of columns names depending on what you've downloaded eg.:
#cols = ["Gold", "Silver", "DXY"]

def returns (data, cols):
    
    data = data["Close"].copy() #copy to avoid an indexing error with original frame
    data.reset_index(inplace = True)
    for colname in cols:
        data[colname+ "_return"] = (data[colname] - data[colname].shift(1)) / data[colname].shift(1)
        data.dropna(0, inplace = True)
    
    return data
    
    

In [ ]:
def returns(

In [ ]:
data = data_recent["Close"].copy()
data.reset_index(inplace=True)
data["DXY Return"] = data["DX-Y.NYB"] - data["DX-Y.NYB"].shift(1) / data["DX-Y.NYB"].shift(1)
data["GC=F Return"] = data["GC=F"] - data["GC=F"].shift(1) / data["GC=F"].shift(1)
data["SI=F Return"] = data["SI=F"] - data["SI=F"].shift(1) / data["SI=F"].shift(1)

data


In [ ]:
#Execute the function above
#Need to rename the columns in data_recent to silver ext first.
cols = ["DX-Y.NYB", "GC=F", "SI=F"]
        
data_recent = returns(data, cols)

In [ ]:
data_recent.rename(columns = {"^GSPC": "S&P", "^GSPC_return":"S&P_return"}, inplace = True)

In [ ]:
data.columns

In [ ]:
data_recent = data[['Date', 'GC=F', 'SI=F', 'DX-Y.NYB', 'GC=F Return',
       'SI=F Return', 'DXY Return']]

In [ ]:
#this only helps with using column names, not data
stock1 = data_recent.columns[1]
stock2 = data_recent.columns[2]
market_name = data_recent.columns[3]
return1 = data_recent.columns[4]
return2 = data_recent.columns[5]
market_return = data_recent.columns[6]

### Calculating Beta

#### Finding Variance and Covariance of Returns for stocks and market to calculate BETA

In [ ]:
#Beta Neutrality
stock1_price = data_recent[stock1].iloc[-1] #most reent price column 1
stock2_price = data_recent[stock2].iloc[-1]

print(f'the latest price of {stock1} is {stock1_price}')
print(f'the latest price of {stock2} is {stock2_price}')

In [ ]:
#To use this you need a dataframe, you need objects that are columns for the other three variables
#nshares is an integer
#if shorting make sure the short data is in the second column

def BETA(data, return1, return2, market_return, nshares, operation):
    covariance = data[[return1, return2, market_return]].cov()
    market_var = covariance.loc[market_return , market_return]
    stock1_cov_to_market = covariance.iloc[0,2] 
    stock2_cov_to_market = covariance.iloc[1,2]
    market_var = covariance.iloc[2,2]
    stock1_Beta = stock1_cov_to_market / market_var
    stock2_Beta = stock2_cov_to_market / market_var
    riskiness = stock1_Beta/stock2_Beta

    stock1_price = data.iloc[-1, 1] #most reent price column 1
    stock2_price = data.iloc[-1, 2] #most recent price column 2

    if operation == 'short':
        adj_val2 = (stock2_price * nshares) * stock2_Beta
        long_shares_stock_1 = adj_val2 / (stock1_Beta * stock1_price)
        stock_1_cost = long_shares_stock_1 * stock1_price
    
        return long_shares_stock_1, stock_1_cost
    
    elif operation == 'long':
        
        adj_val1 = (stock1_price * nshares) * stock2_Beta
        short_shares_stock_2 = adj_val1 / (stock2_Beta * stock2_price)
        stock2_cost = short_shares_stock_2 * stock2_price
        
        return short_shares_stock_2, stock2_cost
    
    else:
        return "Well that hasn't worked, check your inputs and make sure you said long or short"
    
    
    return stock1_Beta, stock2_Beta, riskiness, long_shares_stock_1, stock1_cost, short_shares_stock_2, stock2_cost         
    

In [ ]:
long_stock_units, long_stock_cost = BETA(data_recent, return1, return2, market_return, 3, "short")

print(f'if you are going short {stock2} then you will go long {stock1} {long_stock_units} units at a cost of {long_stock_cost}')

### Beta Weighting

This model asserts that a stock's return is based on its Beta

<img src = "CAPM.png" alt="CAPM Model" width=300 height =300 >


#### Beta Nutrality

Beta stock1 X Price stock 1 X nShares stock 1 == Beta stock2 X Price stock 2 X nShares stock 2

i.e the beta of apple X the market value of apple should equal the beta of msft X the market value os msft

In [ ]:
stock1_price = data_recent.iloc[-1, 1] #most reent price column 1
stock2_price = data_recent.iloc[-1, 2]

In [ ]:
print(f'{stock1_price} and this is stock 2 price {stock2_price}')

In [ ]:
#Calculate first stock based on second stock beta and shares to short

def BetaShort (price1, price2, beta1, beta2, nshort_shares2):
    adj_val2 = (price2 * nshort_shares2) * beta2
    long_shares1 = adj_val2 / (beta1 * price1)
    stock1_cost = long_shares1 * price1
    
    print(f'the amount to go long is {long_shares1} shares which will cost {stock_1_cost}')
    

In [ ]:
#Calculate second stock short based on first stock long

def BetaLong (price1, price2, beta1, beta2, nlong_shares1):
    adj_val1 = (price1 * nlong_shares1) * beta1
    short_shares2 = adj_val1 / (beta2 * price2)
    stock2_cost = short_shares2 * price2
    
    print(f'the amount to go short is {short_shares2} shares of stock 2 which will cost {stock2_cost}')

In [ ]:
BetaLong(2475, 2024, stock1_Beta, stock2_Beta, 2)

I wish to sell short stock2 100 shares so how many shares of stock 1 do i need to buy

In [ ]:
short_shares_2 = 3

In [ ]:
adj_market_val_2 = (stock2_price * short_shares_2) * stock2_Beta
print(f'the amount to go short {stock2} is ${adj_market_val_2} ')

In [ ]:
long_shares = adj_market_val_2 / (stock1_Beta * stock1_price)
stock_1_cost = long_shares * stock1_price
print(f'the amount to go long {stock1} is {long_shares} shares which will cost {stock_1_cost}')

In [ ]:
long_shares * stock1_price * stock1_Beta

In [ ]:
short_shares_2 * stock2_price * stock2_Beta

#### Position size

#### Correlation

In [ ]:
data_history.corr(), 

In [ ]:
data_recent.corr()

In [ ]:
data_last_week.corr()

### Plotting 

In [ ]:
Pairs = data.split(", ") # indexes by , separated value
Pairs

In [ ]:
col2 = data_recent.columns[1]
col3 = data_recent.columns[2]

In [ ]:
y = data_recent[Pairs[0]]
z = data_recent[Pairs[1]]
x = data_recent["Date"]

In [ ]:
y = data_recent.iloc[:, 1] #data from second column
z = data_recent.iloc[:, 2] #data from third column
x = data_recent.iloc[:, 0] #date data from zero index column

In [ ]:
history_min_date = data_history["Date"].min().date()
history_max_date = data_history["Date"].max().date()
recent_min_date = data_recent["Date"].min().date() #add .date() to remove the time stamp
recent_max_date = data_recent["Date"].max().date() 
last_week_min_date = data_last_week["Date"].min().date()
last_week_max_date = data_last_week["Date"].max().date() 

In [ ]:
fig, ax1 = plt.subplots(figsize = (15,10))
plt.title(f"Recent Price Trend from {recent_min_date} to {recent_max_date}",fontdict = {'fontsize': 20})
ax2 = ax1.twinx()

color = 'tab:green'
ax1.plot(x, y, color = color)
ax1.set_xlabel('Date')
ax1.set_ylabel({col2}, color = color, )

ax2.plot(x,z, 'b-', label = {col2})
ax2.set_ylabel({col3}, color = 'b')

### Kalman Filtering

Kalman filtering is a dynamic way of weighting the pairs trade, it requires the trader to update it and adjust positions as conditions change. It has covariance baked in and is weighted forward. It doesnt require rolling windows.
Beta weighting is more static but takes into account the benchmark index and removes it. Kalman is seen as harder and more complex to implement although seems of equal effort to me.

In [ ]:
himx = data_recent["HIMX"]
apple = data_recent["AAPL"]

In [ ]:
print(himx.isna().sum())

In [ ]:
from pykalman import KalmanFilter

In [ ]:
#taken from coursera
kf = KalmanFilter(transition_matrices = [1],
                  observation_matrices = [1],
                  initial_state_mean = 0, #have to have a start value, this changes later
                  initial_state_covariance = 1,
                  observation_covariance = 1,
                  transition_covariance = 0.01)

In [ ]:
#Use values of price to get rolling meeans nased on the kalman equation which attempts to remove noise from the price data
state_means,_ = kf.filter(himx.values)
state_means = pd.Series(state_means.flatten(), index=himx.index) 

In [ ]:
#compute rolling average
mean50 = himx.rolling(window =50).mean()
mean100 = himx.rolling(window =100).mean()

#plot orignal data and means
plt.plot(state_means)
plt.plot(himx)
plt.plot(mean50)
plt.plot(mean100)
plt.title("Experimenting with Kalman mean to remove price noise")
plt.legend(['Kalman Estimate', "Himx Price", "50 MA", "100 MA"])
plt.xlabel("Day")
plt.ylabel('Price')


In [ ]:
spread = apple - himx

In [ ]:
#Using Kalman as a hedge ratio instead of beta weighting

# Reshape spread.values to a 2D array (n_samples, 1)
spread_reshaped = spread.values.reshape(-1, 1)

# Calculate the hedge ratio using the Kalman filter
kf = KalmanFilter(transition_matrices=[1], observation_matrices=np.eye(1), observation_covariance=1)
state_means, _ = kf.filter(spread_reshaped)
hedge_ratio = state_means.flatten()

# Calculate the number of shares to trade for each stock based on a capital of $10,000
capital_kf = 10000  # USD
aapl_price = apple.iloc[-1]  # Assuming last price for AAPL
himx_price = himx.iloc[-1]  # Assuming last price for HIMX

# Calculate the number of shares for each stock based on the hedge ratio and capital
aapl_shares = (capital_kf / (aapl_price + hedge_ratio[-1] * himx_price))
himx_shares = hedge_ratio[-1] * aapl_shares

print(f"Estimated number of AAPL shares to trade: {aapl_shares:.2f}")
print(f"Estimated number of HIMX shares to trade: {himx_shares:.2f}")


In [ ]:
#take a look at pairs selection from the labe and reprodice the code here, it should be about the beta weighting.
#find negatively correlated stocks
#Find negatively correlated funds.
